In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# 1. Dataset Configuration (With Resize for ViT)

class ViTLensingDataset(Dataset):
    def __init__(self, data_dir, limit_per_class=800, img_size=64):
        self.filepaths = []
        self.labels = []
        self.img_size = img_size
        
        classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('.')]
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        
        for cls_name, idx in self.class_to_idx.items():
            files = glob.glob(os.path.join(data_dir, cls_name, '*.npy'))[:limit_per_class]
            self.filepaths.extend(files)
            self.labels.extend([idx] * len(files))

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = np.load(self.filepaths[idx])
        if len(img.shape) == 2:
            img = np.expand_dims(img, axis=0)
            
        tensor_img = torch.tensor(img, dtype=torch.float32)
        
        # Resize to 64x64 to make ViT manageable on CPU
        tensor_img = F.interpolate(tensor_img.unsqueeze(0), size=(self.img_size, self.img_size), mode='bilinear', align_corners=False).squeeze(0)
        
        # Log-Scale + Min-Max Normalization
        tensor_img = tensor_img - tensor_img.min()
        tensor_img = torch.log1p(tensor_img)
        t_max = tensor_img.max()
        if t_max > 0:
            tensor_img = tensor_img / t_max
            
        return tensor_img, torch.tensor(self.labels[idx], dtype=torch.long)

# 2. Vision Transformer (ViT) Architecture

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_channels=1, embed_dim=64):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_channels * patch_size * patch_size
        
        # Using a pure Linear layer instead of Conv2d to strictly follow "No CNN" rules
        self.proj = nn.Linear(self.patch_dim, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        # Extract patches mathematically: (B, C, H, W) -> (B, num_patches, patch_dim)
        x = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        x = x.contiguous().view(B, C, -1, self.patch_size, self.patch_size)
        x = x.permute(0, 2, 1, 3, 4).contiguous()
        x = x.view(B, -1, self.patch_dim)
        
        # Project patches to embedding dimension
        x = self.proj(x)
        return x

class MicroViT(nn.Module):
    def __init__(self, img_size=64, patch_size=8, num_classes=3, embed_dim=64, depth=4, heads=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 1, embed_dim)
        num_patches = self.patch_embed.num_patches
        
        # Learnable class token and positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=heads, dim_feedforward=embed_dim*2, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        
        # Classification Head
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        # Prepend class token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        # Add positional embedding
        x = x + self.pos_embed
        
        # Pass through Transformer
        x = self.transformer(x)
        
        # Extract the class token output for classification
        cls_out = x[:, 0]
        cls_out = self.norm(cls_out)
        return self.head(cls_out)

# 3. Training Pipeline

def main():
    data_dir = "dataset_extraido/dataset/train"
    dataset = ViTLensingDataset(data_dir, limit_per_class=800, img_size=64)
    
    train_size = int(0.9 * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = random_split(dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
    
    device = torch.device("cpu")
    model = MicroViT(num_classes=len(dataset.class_to_idx)).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    epochs = 10
    print(f"Starting Vision Transformer (ViT) training on {device}...")
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_loss)
        print(f"Epoch [{epoch}/{epochs}] | Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.5f}")

    # 4. Evaluation & ROC

    model.eval()
    all_targets, all_probs = [], []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            
            all_targets.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    classes_list = list(dataset.class_to_idx.keys())
    binarized_targets = label_binarize(all_targets, classes=list(dataset.class_to_idx.values()))
    
    plt.figure(figsize=(8, 6))
    for i in range(len(classes_list)):
        fpr, tpr, _ = roc_curve(binarized_targets[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{classes_list[i]} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Specific Test V: Vision Transformer ROC Curve')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

    torch.save(model.state_dict(), "Specific_Test_V_ViT.pth")
    print("Model weights saved to 'Specific_Test_V_ViT.pth'")

if __name__ == '__main__':
    main()